# 台灣50正2（00631L）快速修復回檔入場策略 — FinLab 回測

大跌之後如果**修復得夠快**，行情大概率會回到舊高，而途中的回檔**淺到不值得等**。
本 notebook 把這個觀察寫成規則，訊號來自**加權指數**，部位開在 **00631L**，
用 `finlab.backtest.sim()` 回測並產生 `report.display()` 的互動報表。

**執行方式**：由上而下依序執行即可。需要 FinLab API token。

---
### 30 秒版本
```
訊號   自高點回檔 ≥10% 後，從谷底起 ≤30 個交易日內收盤補回 ≥60% 的跌幅
進場   訊號確認後次一交易日一次買足目標水位，不等回檔
防守   收盤跌破「補回一半」的線（含 0.5% 緩衝）→ 全部出場
       收盤跌破谷底 → 全部出場
出場   前高不賣；創高後啟動移動停利，自波段最高收盤回檔 8% → 出清
```

> ⚠️ 這是研究專案，**不是投資建議**。00631L 是 2 倍槓桿工具，
> 指數 −1% ≈ ETF −2%，另有波動耗損、內扣與折溢價風險。
> 回測期間 2014-10 起僅約 7 次訊號，樣本數不足以支撐統計結論。


## 1. 安裝套件


In [ ]:
!pip install -q finlab


## 2. 登入 FinLab

執行後貼上你的 API token（不會顯示在畫面上）。
token 可在 [FinLab 會員頁面](https://ai.finlab.tw/) 取得。


In [ ]:
import getpass, os, warnings
import finlab

token = os.environ.get('FINLAB_API_TOKEN') or getpass.getpass('FinLab API token: ')
with warnings.catch_warnings():
    warnings.simplefilter('ignore', DeprecationWarning)
    finlab.login(token)


## 3. 寫入策略程式碼

以下是 `tw_backdraw` 套件的完整原始碼，直接寫成檔案後 import ——
與專案回測使用的是同一份程式，不是簡化版。


In [ ]:
import pathlib

PKG = pathlib.Path('tw_backdraw')
PKG.mkdir(exist_ok=True)
SOURCES = {}

SOURCES['bars'] = '"""日線資料結構與讀檔。"""\n\nfrom __future__ import annotations\n\nimport csv\nfrom dataclasses import dataclass\nfrom datetime import date, datetime\nfrom pathlib import Path\n\n\n@dataclass(frozen=True)\nclass Bar:\n    d: date\n    open: float\n    high: float\n    low: float\n    close: float\n\n    @property\n    def iso(self) -> str:\n        return self.d.isoformat()\n\n\ndef _parse_date(raw: str) -> date:\n    raw = raw.strip()\n    for fmt in ("%Y-%m-%d", "%Y/%m/%d", "%Y%m%d"):\n        try:\n            return datetime.strptime(raw, fmt).date()\n        except ValueError:\n            continue\n    raise ValueError(f"無法解析日期: {raw!r}")\n\n\ndef load_csv(path: str | Path) -> list[Bar]:\n    """讀取日線 CSV。\n\n    必要欄位: date, close。open/high/low 缺漏時以 close 補齊，\n    這樣只有收盤價的資料集也能直接跑（策略訊號全部以收盤價判定）。\n    """\n    bars: list[Bar] = []\n    with open(path, newline="", encoding="utf-8-sig") as fh:\n        for row in csv.DictReader(fh):\n            row = {(k or "").strip().lower(): (v or "").strip() for k, v in row.items()}\n            if not row.get("date") or not row.get("close"):\n                continue\n            close = float(row["close"].replace(",", ""))\n\n            def pick(key: str) -> float:\n                val = row.get(key, "")\n                return float(val.replace(",", "")) if val else close\n\n            bars.append(\n                Bar(\n                    d=_parse_date(row["date"]),\n                    open=pick("open"),\n                    high=pick("high"),\n                    low=pick("low"),\n                    close=close,\n                )\n            )\n    bars.sort(key=lambda b: b.d)\n    if not bars:\n        raise ValueError(f"{path} 沒有可用的日線資料")\n    return bars\n'
SOURCES['config'] = '"""策略參數。\n\n所有可調參數集中於此，方便做敏感度測試。\n預設值對應貼文中的歷史統計（12 國、近百年、174 次樣本）。\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\n\n\n@dataclass(frozen=True)\nclass SetupConfig:\n    """「快速修復」劇本的辨識條件。"""\n\n    # 先要有一段像樣的回檔才談得上修復，貼文樣本為 -16%\n    min_drawdown: float = 0.10\n    # 從谷底起算，補回跌幅的多少比例才算「補回來了」。\n    # 貼文的原始值是 0.75；預設改用 grid search 以「總報酬 ÷ 最大回檔」選出的 0.60。\n    repair_fraction: float = 0.60\n    # 補回必須在幾個交易日內完成。\n    # 貼文的原始值是 15（對應「89% 創高」那一組）；預設改用 0.60/30 這組。\n    max_repair_bars: int = 30\n    # 訊號有效期；超過仍未創高也未失效就自然過期\n    setup_expiry_bars: int = 120\n    # 修復視窗過期後，把參考高點重新錨定到「谷底之後的波段高」。\n    # 關掉的話，參考高點會一直釘在舊高，直到指數重新站上它為止 ——\n    # 台股 2000 年頭部之後花了 17.3 年才收復 10,202，等於中間完全看不到訊號。\n    reanchor_on_expiry: bool = True\n\n\n@dataclass(frozen=True)\nclass LevelConfig:\n    """由 P（前高）與 T（谷底）推出的關鍵價位。"""\n\n    # 主防線：補回一半\n    half_line_ratio: float = 0.50\n    # 警戒線：貼文的原始值是 0.382（台股 42916）。\n    # 預設改用 0.500 —— 這會讓警戒線與主防線重合，\n    # 等於「跌破補回一半的線（含 0.5% 緩衝）就全部出場」，是明顯更緊的停損。\n    warn_line_ratio: float = 0.500\n    # 允許對主防線的假跌破緩衝（貼文：一半案例跌破不到 0.5%）\n    half_line_buffer: float = 0.005\n\n\n@dataclass(frozen=True)\nclass EntryConfig:\n    """分批進場梯。\n\n    核心前提：歷史上回檔中位數只有 2.9%、四分之三不超過 5%，\n    所以「等深回檔」的期望值是負的 —— 底倉先上車，回檔才是加碼機會。\n    """\n\n    # 訊號確認後隔日直接建立的底倉權重。\n    # 貼文原意是 0.40（保留兩段回檔加碼空間）；預設改用 1.00，\n    # 也就是訊號一確認就把目標水位一次買足，不留加碼梯。\n    base_weight: float = 1.00\n    # (自訊號後波段高點的回檔幅度, 加碼權重)\n    pullback_ladder: tuple[tuple[float, float], ...] = ((0.03, 0.30), (0.05, 0.30))\n    # 幾個交易日內若都沒等到回檔，就以市價補齊剩餘部位（不參與才是最大風險）。\n    # 註：預設 base_weight=1.00 時沒有加碼梯，本參數不影響任何決策；\n    # grid search 對它的取值也近乎均勻分布（10/20/40 各約三分之一），維持 20 為中性值。\n    fill_timeout_bars: int = 20\n    # 突破前高後補齊剩餘部位。\n    # 預設 True：訊號觸發時距前高通常只剩 3~4%，等不到回檔梯就先創高是常態，\n    # 若此時取消加碼，實際部署會長期停在底倉水位（實測平均僅 24%）。\n    # 往上買的單位風險較高，引擎會按停損距離自動縮小這一段的權重。\n    breakout_fills_remainder: bool = True\n\n\n@dataclass(frozen=True)\nclass ExitConfig:\n    """出場與風控。"""\n\n    # 收盤跌破警戒線 → 減碼比例。\n    # 貼文原意是 0.50（砍一半、保留船票）；預設改用 1.00，也就是跌破就全部出場。\n    # 這會降低勝率、提高總報酬與報酬÷回檔（見 docs/strategy.md §11）。\n    warn_derisk_fraction: float = 1.00\n    # 收盤跌破谷底 → 全出（貼文中 11% 的失敗案例都是大熊市開場）\n    hard_stop_at_trough: bool = True\n    # 觸及前高後先落袋的比例。\n    # 預設 0：「離前高很近，本身不是賣出的理由」——前高只是統計上的高機率目標，\n    # 不是出場訊號；獲利全部交給停利機制處理。設 1/3 可改回分批落袋。\n    target_take_fraction: float = 0.0\n\n    # 創高之後用哪一種停利：\n    #   "trail"       自創高後最高收盤回檔 trail_drawdown → 出場\n    #   "ma_ratchet"  停利價 = max(前高, MA)。創高後先把前高當出場線，\n    #                 等 MA 爬過前高，就改看 MA 跌破 —— 均線只會把出場線往上推。\n    #   "both"        兩者取較緊（較高）的那一條\n    exit_mode: str = "trail"\n    # ma_ratchet / both 使用的均線天期（以加權指數收盤計）\n    ma_period: int = 20\n    # trail / both 使用的回檔幅度（以指數計）\n    trail_drawdown: float = 0.08\n\n\n@dataclass(frozen=True)\nclass SizingConfig:\n    """部位規模（標的為 2 倍槓桿 ETF，必須以指數停損距離反推）。"""\n\n    # 單筆交易願意承受的權益風險（兩段式停損全走完的預期損失）\n    risk_per_trade: float = 0.08\n    # 標的槓桿倍數（台灣50正2 = 2）\n    leverage: float = 2.0\n    # 最高持股水位（佔權益比例）\n    max_weight: float = 1.0\n    # 停損距離至少視為這麼大，避免訊號日離谷底太近而算出過大部位\n    min_stop_distance: float = 0.05\n\n\n@dataclass(frozen=True)\nclass CostConfig:\n    """台股 ETF 交易成本與槓桿 ETF 的內扣損耗。"""\n\n    # 券商手續費 0.1425% × 折扣\n    fee_rate: float = 0.001425\n    fee_discount: float = 0.60\n    # ETF 賣出證交稅 0.1%\n    tax_rate: float = 0.001\n    # 槓桿 ETF 年化內扣（管理費 + 期貨轉倉/避險成本）\n    annual_carry: float = 0.012\n    trading_days: int = 252\n\n    @property\n    def buy_cost(self) -> float:\n        return self.fee_rate * self.fee_discount\n\n    @property\n    def sell_cost(self) -> float:\n        return self.fee_rate * self.fee_discount + self.tax_rate\n\n    @property\n    def daily_carry(self) -> float:\n        return self.annual_carry / self.trading_days\n\n\n@dataclass(frozen=True)\nclass StrategyConfig:\n    setup: SetupConfig = field(default_factory=SetupConfig)\n    levels: LevelConfig = field(default_factory=LevelConfig)\n    entry: EntryConfig = field(default_factory=EntryConfig)\n    exit: ExitConfig = field(default_factory=ExitConfig)\n    sizing: SizingConfig = field(default_factory=SizingConfig)\n    cost: CostConfig = field(default_factory=CostConfig)\n\n\n# ---------------------------------------------------------------------------\n# 預設組（PRESETS）\n#\n# 以下各組是 648,000 組 grid search + 樣本外驗證（scripts/grid_search.py、\n# scripts/walk_forward.py）的產物。**預設是 tuned**，也就是以總報酬為目標\n# 選出來的那一組；忠於貼文的原始設定保留在 `post`。\n# 兩者的取捨與已知風險記在 docs/strategy.md §11，換組合前請先讀。\n# ---------------------------------------------------------------------------\n\n#: 以「總報酬 ÷ 最大回檔」為目標選出的參數（本專案的預設）。\n#: 放寬訊號定義（30 日 / 補回 60%）、訊號確認即滿倉、跌破主防線全數出場。\n#: 樣本內 21 筆、勝率 57%、總報酬 +2232%、最大回檔 -35.8%、比值 62.4。\n#: 樣本外驗證中，這個目標函數的表現優於直接用總報酬（見 docs/strategy.md §11）。\nTUNED_CONFIG = StrategyConfig()\n\nDEFAULT_CONFIG = TUNED_CONFIG\n\n#: 忠於貼文的原始設定：15 日內補回 75%、底倉四成留加碼梯、\n#: 38.2% 回補位減碼一半。樣本內 8 筆、勝率 50%、總報酬 +31.4%、最大回檔 -20.4%。\nPOST_CONFIG = StrategyConfig(\n    setup=SetupConfig(repair_fraction=0.75, max_repair_bars=15),\n    levels=LevelConfig(warn_line_ratio=0.382),\n    entry=EntryConfig(base_weight=0.40),\n    exit=ExitConfig(warn_derisk_fraction=0.50),\n)\n\n#: 調校過的訊號 + MA40 棘輪出場。用報酬換勝率與較小的回檔：\n#: 樣本內 21 筆、勝率 67%、總報酬 +459%、回檔 -35%。\nBALANCED_CONFIG = StrategyConfig(\n    exit=ExitConfig(warn_derisk_fraction=1.0, exit_mode="ma_ratchet", ma_period=40),\n)\n\n#: 全網格勝率最高的一組。**它是靠關掉兩道停損換來的**，總報酬遠低於預設組。\n#: 列在這裡是為了讓「最大化勝率」的後果可以被重現，不是建議值。\nWINRATE_CONFIG = StrategyConfig(\n    setup=SetupConfig(min_drawdown=0.13),\n    levels=LevelConfig(warn_line_ratio=0.382),\n    entry=EntryConfig(fill_timeout_bars=40),\n    exit=ExitConfig(warn_derisk_fraction=0.0, hard_stop_at_trough=False,\n                    exit_mode="ma_ratchet", ma_period=40),\n)\n\n#: 美股版（S&P 500 訊號 → UPRO 3x 執行）以「總報酬 ÷ 最大回檔」選出的參數。\n#: 與台股版的差異：回檔門檻 7%（S&P 500 在 FinLab 涵蓋的 10.6 年裡 ≥10% 的\n#: 回檔太少）、移動停利放寬到 10%（3 倍槓桿的波動較大）。\n#: 樣本內 8 筆、勝率 62%、總報酬 +627%、最大回檔 -37.5%、比值 16.7。\n#: ⚠️ min_drawdown 落在搜尋網格的下界，最佳值可能在網格之外 —— 見 docs/strategy.md §12。\nUS_TUNED_CONFIG = StrategyConfig(\n    setup=SetupConfig(min_drawdown=0.07, repair_fraction=0.60, max_repair_bars=30),\n    levels=LevelConfig(warn_line_ratio=0.500),\n    entry=EntryConfig(base_weight=1.0, fill_timeout_bars=10),\n    exit=ExitConfig(warn_derisk_fraction=1.0, exit_mode="trail", trail_drawdown=0.10),\n    sizing=SizingConfig(leverage=3.0),\n    cost=CostConfig(fee_rate=0.0, fee_discount=1.0, tax_rate=0.0, annual_carry=0.0091),\n)\n\n#: NASDAQ 版（^IXIC 訊號 → TQQQ 3x 執行）以「總報酬 ÷ 最大回檔」選出的參數。\n#: 樣本內 8 筆、勝率 88%、總報酬 +933%、最大回檔 -30.8%、比值 30.3。\n#:\n#: ⚠️ trail_drawdown=0.12 是**尖峰而非平台**：鄰近值的比值為 10%→19.0、\n#: 12%→30.3、15%→15.7，且交易數從 12 筆掉到 8 筆。這是過擬合的典型特徵，\n#: 較穩健的鄰居是 trail=0.08（16 筆、比值 21.3）。詳見 docs/strategy.md §13。\nNQ_TUNED_CONFIG = StrategyConfig(\n    setup=SetupConfig(min_drawdown=0.07, repair_fraction=0.60, max_repair_bars=30),\n    levels=LevelConfig(warn_line_ratio=0.382),\n    entry=EntryConfig(base_weight=1.0, fill_timeout_bars=10),\n    exit=ExitConfig(warn_derisk_fraction=1.0, exit_mode="trail", trail_drawdown=0.12),\n    sizing=SizingConfig(leverage=3.0),\n    cost=CostConfig(fee_rate=0.0, fee_discount=1.0, tax_rate=0.0, annual_carry=0.0095),\n)\n\n#: 同上但把移動停利改成鄰域穩健的 8%：16 筆、勝率 62%、+570%、-26.8%、比值 21.3。\nNQ_ROBUST_CONFIG = StrategyConfig(\n    setup=NQ_TUNED_CONFIG.setup, levels=NQ_TUNED_CONFIG.levels,\n    entry=NQ_TUNED_CONFIG.entry,\n    exit=ExitConfig(warn_derisk_fraction=1.0, exit_mode="trail", trail_drawdown=0.08),\n    sizing=NQ_TUNED_CONFIG.sizing, cost=NQ_TUNED_CONFIG.cost,\n)\n\nPRESETS: dict[str, StrategyConfig] = {\n    "tuned": TUNED_CONFIG,\n    "post": POST_CONFIG,\n    "balanced": BALANCED_CONFIG,\n    "winrate": WINRATE_CONFIG,\n    "us_tuned": US_TUNED_CONFIG,\n    "nq_tuned": NQ_TUNED_CONFIG,\n    "nq_robust": NQ_ROBUST_CONFIG,\n}\n'
SOURCES['levels'] = '"""由「前高 P」與「谷底 T」推導出的關鍵價位。\n\n台股當前實例：P = 47742、T = 39933、跌幅 R = 7809 點\n    主防線 (50%)  = 43837   ← 貼文的 43800\n    警戒線 (38.2%) = 42916\n    失效線        = 39933\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nfrom .config import LevelConfig\n\n\n@dataclass(frozen=True)\nclass Levels:\n    peak: float\n    trough: float\n    half_line: float\n    half_line_with_buffer: float\n    warn_line: float\n    invalidation: float\n\n    @property\n    def drop(self) -> float:\n        return self.peak - self.trough\n\n    @property\n    def drop_pct(self) -> float:\n        return self.drop / self.peak\n\n    def repair_fraction(self, price: float) -> float:\n        """價格相當於補回跌幅的幾成。"""\n        if self.drop <= 0:\n            return 0.0\n        return (price - self.trough) / self.drop\n\n    def zone(self, price: float) -> str:\n        """把價格歸類到操作分區。"""\n        if price >= self.peak:\n            return "breakout"       # 已創高：轉移動停利\n        if price >= self.half_line:\n            return "healthy"        # 劇本正常：回檔即加碼\n        if price >= self.half_line_with_buffer:\n            return "buffer"         # 假跌破容忍區：不加碼、不減碼\n        if price >= self.warn_line:\n            return "caution"        # 跌破主防線：停止加碼\n        if price >= self.invalidation:\n            return "warning"        # 跌破 38.2%：減碼\n        return "invalidated"        # 跌破谷底：另一個故事，全出\n\n\ndef build_levels(peak: float, trough: float, cfg: LevelConfig) -> Levels:\n    if peak <= trough:\n        raise ValueError("前高必須高於谷底")\n    drop = peak - trough\n    half = trough + cfg.half_line_ratio * drop\n    return Levels(\n        peak=peak,\n        trough=trough,\n        half_line=half,\n        half_line_with_buffer=half * (1 - cfg.half_line_buffer),\n        warn_line=trough + cfg.warn_line_ratio * drop,\n        invalidation=trough,\n    )\n'
SOURCES['setup'] = '"""辨識「快速修復」劇本。\n\n流程（全部以收盤價、可即時判定，不使用未來資料）：\n    1. 從滾動高點 P 起算，收盤跌破 P×(1-10%) → 進入回檔追蹤\n    2. 追蹤期間持續更新谷底 T（創更低就更新，計時歸零）\n    3. 自 T 起 N 個交易日內，收盤補回跌幅 ≥ 75% → 觸發訊號\n       （N ≤ 15 對應貼文中「89% 會先回到舊高點」的那一組）\n    4. 超過 15 日才補回 → 這一段作廢（那一組成功率只剩 48%，跟丟銅板一樣），\n       參考高點改錨到谷底之後的波段高，重新開始找下一組 P/T\n\n第 4 步的改錨很關鍵。少了它，參考高點會一直釘在舊高直到指數重新站上為止 ——\n台股 2000 年頭部之後花了 17.3 年才收復 10,202，中間包含 2008、2015、2020\n在內的所有回檔修復都會被那個舊高遮蔽，偵測器等於瞎掉。\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom datetime import date\n\nfrom .bars import Bar\nfrom .config import SetupConfig\n\n\n@dataclass(frozen=True)\nclass Episode:\n    """一段 ≥10% 的回檔，以及它為什麼（沒）觸發訊號。"""\n\n    peak: float\n    peak_date: date\n    peak_index: int\n    trough: float\n    trough_date: date\n    trough_index: int\n    lower_lows: int              # 追蹤期間破底幾次（每次都讓計時歸零）\n    best_in_window: float        # 15 日視窗內補回的最高比例\n    best_in_window_bars: int     # 上述最佳值出現在谷底後第幾日\n    fired: bool\n    end_index: int\n    end_date: date\n\n    @property\n    def drop_pct(self) -> float:\n        return (self.trough - self.peak) / self.peak\n\n    def reason(self, cfg: SetupConfig) -> str:\n        if self.fired:\n            return f"✅ {self.best_in_window_bars} 日補回 {self.best_in_window:.0%}"\n        return (f"❌ {cfg.max_repair_bars} 日內只補回 {self.best_in_window:.0%}"\n                f"（第 {self.best_in_window_bars} 日最佳）")\n\n\n@dataclass(frozen=True)\nclass FastRepairSetup:\n    peak: float\n    peak_date: date\n    trough: float\n    trough_date: date\n    trough_index: int\n    trigger_index: int\n    trigger_date: date\n    trigger_close: float\n    bars_to_repair: int\n    repair_fraction: float\n\n    @property\n    def drop_pct(self) -> float:\n        return (self.peak - self.trough) / self.peak\n\n    def describe(self) -> str:\n        return (\n            f"{self.trough_date} 谷底 {self.trough:,.0f}（自 {self.peak_date} 高點 "\n            f"{self.peak:,.0f} 回檔 {self.drop_pct:.1%}），"\n            f"{self.bars_to_repair} 個交易日補回 {self.repair_fraction:.0%}，"\n            f"{self.trigger_date} 觸發訊號 @ {self.trigger_close:,.0f}"\n        )\n\n\ndef scan_episodes(bars: list[Bar], cfg: SetupConfig) -> list[Episode]:\n    """列出所有 ≥ `min_drawdown` 的回檔段落，含觸發與未觸發的原因。\n\n    `detect_setups()` 就是取這裡面 `fired=True` 的那些，所以兩者不會不一致。\n    """\n    episodes: list[Episode] = []\n    if not bars:\n        return episodes\n\n    peak, peak_i = bars[0].close, 0\n    trough, trough_i = bars[0].close, 0\n    state = "normal"\n    lower_lows = 0\n    win_best, win_best_n = 0.0, 0\n\n    def flush(i: int, fired: bool) -> None:\n        episodes.append(Episode(\n            peak=peak, peak_date=bars[peak_i].d, peak_index=peak_i,\n            trough=trough, trough_date=bars[trough_i].d, trough_index=trough_i,\n            lower_lows=lower_lows, best_in_window=win_best,\n            best_in_window_bars=win_best_n, fired=fired,\n            end_index=i, end_date=bars[i].d,\n        ))\n\n    for i, bar in enumerate(bars):\n        c = bar.close\n\n        if state == "normal":\n            if c > peak:\n                peak, peak_i = c, i\n            elif c <= peak * (1 - cfg.min_drawdown):\n                state = "drawdown"\n                trough, trough_i = c, i\n                lower_lows, win_best, win_best_n = 0, 0.0, 0\n            continue\n\n        if state == "drawdown":\n            if c < trough:\n                # 創更低點：谷底、計時與視窗內最佳進度一起重設\n                trough, trough_i = c, i\n                lower_lows += 1\n                win_best, win_best_n = 0.0, 0\n                continue\n\n            elapsed = i - trough_i\n            frac = (c - trough) / (peak - trough) if peak > trough else 0.0\n            if elapsed <= cfg.max_repair_bars and frac > win_best:\n                win_best, win_best_n = frac, elapsed\n\n            if frac >= cfg.repair_fraction and elapsed <= cfg.max_repair_bars:\n                flush(i, fired=True)\n                state = "engaged"\n                continue\n\n            if elapsed > cfg.max_repair_bars:\n                # 修復視窗到期，這一段作廢\n                flush(i, fired=False)\n                if cfg.reanchor_on_expiry:\n                    seg = bars[trough_i:i + 1]\n                    k = max(range(len(seg)), key=lambda j: seg[j].close)\n                    peak, peak_i = seg[k].close, trough_i + k\n                    state = "normal"\n                    continue\n                # 不改錨：等指數重新站上舊高才解除\n                state = "expired"\n                continue\n\n            if c > peak:\n                flush(i, fired=False)\n                state, peak, peak_i = "normal", c, i\n            continue\n\n        if state == "expired":\n            if c > peak:\n                state, peak, peak_i = "normal", c, i\n            continue\n\n        if state == "engaged":\n            last = episodes[-1]\n            if (c > peak or c < last.trough\n                    or (i - last.end_index) > cfg.setup_expiry_bars):\n                state = "normal"\n                if c > peak:\n                    peak, peak_i = c, i\n            continue\n\n    return episodes\n\n\ndef detect_setups(bars: list[Bar], cfg: SetupConfig) -> list[FastRepairSetup]:\n    """掃描整段歷史，回傳所有觸發過的快速修復訊號。"""\n    return [\n        FastRepairSetup(\n            peak=e.peak, peak_date=e.peak_date,\n            trough=e.trough, trough_date=e.trough_date, trough_index=e.trough_index,\n            trigger_index=e.end_index, trigger_date=e.end_date,\n            trigger_close=bars[e.end_index].close,\n            bars_to_repair=e.best_in_window_bars,\n            repair_fraction=e.best_in_window,\n        )\n        for e in scan_episodes(bars, cfg) if e.fired\n    ]\n'
SOURCES['leveraged'] = '"""台灣50正2（00631L）的價格模型。\n\n實務上訊號來自加權指數，執行卻在槓桿 ETF 上，兩者不是同一條線：\n\n* 00631L 追蹤的是「台灣50指數單日報酬 2 倍」，不是加權指數，\n  但兩者日報酬相關性長期在 0.95 以上，訊號層面可互用。\n* 2 倍是「單日」複製，路徑相依：盤整盤會有波動耗損，\n  單邊上漲則會優於 2 倍。\n* 內扣（管理費 + 期貨轉倉/避險）約年化 1%～1.5%，逐日侵蝕淨值。\n\n沒有實際 00631L 日線時，用本模組由指數日報酬合成一條可回測的淨值路徑；\n有實際日線就直接餵真實價格，模型只用來做對照。\n"""\n\nfrom __future__ import annotations\n\nfrom .bars import Bar\nfrom .config import CostConfig\n\n\ndef synth_leveraged_path(\n    bars: list[Bar],\n    cost: CostConfig,\n    leverage: float = 2.0,\n    start_price: float = 100.0,\n) -> list[float]:\n    """由指數日線合成 2 倍槓桿 ETF 的收盤淨值序列。"""\n    path = [start_price]\n    for prev, cur in zip(bars, bars[1:]):\n        r = cur.close / prev.close - 1.0\n        nav = path[-1] * (1.0 + leverage * r - cost.daily_carry)\n        # 槓桿 ETF 淨值不會歸零/轉負，但單日 -50% 指數已超出任何現實情境；\n        # 保底避免回測數值爆掉。\n        path.append(max(nav, 1e-6))\n    return path\n\n\ndef decay_estimate(index_return: float, realized_vol: float, days: int, cost: CostConfig,\n                   leverage: float = 2.0) -> float:\n    """粗估持有 N 日後，槓桿 ETF 相對「指數報酬 × 2」的落差。\n\n    近似式: 2x 報酬 ≈ L·r - 0.5·L·(L-1)·σ²·(days/252) - carry·days/252\n    用來提醒：這個劇本必須是「快速、單邊」的行情才值得用槓桿工具。\n    """\n    variance_drag = 0.5 * leverage * (leverage - 1.0) * (realized_vol ** 2) * (days / cost.trading_days)\n    carry_drag = cost.daily_carry * days\n    return leverage * index_return - variance_drag - carry_drag\n'
SOURCES['engine'] = '"""策略執行引擎（狀態機）。\n\n判斷一律用收盤價，成交一律落在下一個交易日，\n避免「當日收盤發訊號、當日收盤成交」這種實務上做不到的假設。\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom datetime import date\n\nfrom .bars import Bar\nfrom .config import DEFAULT_CONFIG, StrategyConfig\nfrom .levels import Levels, build_levels\nfrom .setup import FastRepairSetup, detect_setups\n\n\n@dataclass\nclass Fill:\n    d: date\n    side: str            # buy / sell\n    reason: str\n    index_price: float\n    etf_price: float\n    weight: float        # 買：佔訊號日權益比例；賣：佔當時持股比例\n    units: float\n    cash_flow: float\n\n    def describe(self) -> str:\n        verb = "買進" if self.side == "buy" else "賣出"\n        return (f"{self.d} {verb} {self.weight:>6.1%} @ ETF {self.etf_price:,.2f} "\n                f"(指數 {self.index_price:,.0f})  {self.reason}")\n\n\n@dataclass\nclass Trade:\n    setup: FastRepairSetup\n    levels: Levels\n    target_weight: float\n    fills: list[Fill] = field(default_factory=list)\n    entry_date: date | None = None\n    exit_date: date | None = None\n    exit_reason: str = "open"\n    equity_at_entry: float = 1.0\n    equity_at_exit: float = 1.0\n    max_adverse_index: float = 0.0   # 進場後指數最大不利波動\n    reached_prior_high: bool = False\n    swing_high: float = 0.0          # 訊號後的最高收盤（回檔梯的基準）\n    pending_ladder: list[tuple[float, float]] = field(default_factory=list)  # 尚未成交的加碼梯\n    bars_held: int = 0\n\n    @property\n    def ret(self) -> float:\n        return self.equity_at_exit / self.equity_at_entry - 1.0\n\n    @property\n    def filled_weight(self) -> float:\n        return sum(f.weight for f in self.fills if f.side == "buy")\n\n\n@dataclass\nclass Result:\n    trades: list[Trade]\n    equity_curve: list[tuple[date, float]]\n    setups: list[FastRepairSetup]\n\n    @property\n    def final_equity(self) -> float:\n        return self.equity_curve[-1][1] if self.equity_curve else 1.0\n\n\ndef _blended_stop_distance(entry_index: float, levels: Levels, cfg: StrategyConfig) -> float:\n    """兩段式停損的預期虧損距離（指數口徑）。\n\n    先在警戒線減碼一半，剩下的在谷底出清，真正的預期損失介於兩者之間。\n    """\n    to_warn = max(entry_index - levels.warn_line, 0.0) / entry_index\n    to_trough = max(entry_index - levels.invalidation, 0.0) / entry_index\n    f = cfg.exit.warn_derisk_fraction\n    blended = f * to_warn + (1.0 - f) * to_trough\n    return max(blended, cfg.sizing.min_stop_distance)\n\n\ndef position_size(entry_index: float, levels: Levels, cfg: StrategyConfig) -> float:\n    """這一筆交易的目標持股水位（佔權益比例）。\n\n    槓桿 ETF 的虧損 ≈ 指數跌幅 × 2，所以指數停損距離越遠、部位必須越小。\n    """\n    expected_loss = cfg.sizing.leverage * _blended_stop_distance(entry_index, levels, cfg)\n    return min(cfg.sizing.max_weight, cfg.sizing.risk_per_trade / expected_loss)\n\n\ndef moving_average(bars: list[Bar], period: int) -> list[float | None]:\n    """加權指數收盤的簡單移動平均；資料不足時為 None。"""\n    out: list[float | None] = []\n    total = 0.0\n    for i, b in enumerate(bars):\n        total += b.close\n        if i >= period:\n            total -= bars[i - period].close\n        out.append(total / period if i >= period - 1 else None)\n    return out\n\n\ndef breakout_stop(cfg: StrategyConfig, prior_high: float, peak_since_breakout: float,\n                  ma: float | None) -> tuple[float, str]:\n    """創高之後的出場線，回傳 (價位, 說明)。"""\n    mode = cfg.exit.exit_mode\n    trail = peak_since_breakout * (1 - cfg.exit.trail_drawdown)\n\n    if mode == "trail":\n        return trail, f"移動停利：自 {peak_since_breakout:,.0f} 回檔 {cfg.exit.trail_drawdown:.0%}"\n\n    # 均線棘輪：先以前高為出場線，等 MA 爬過前高才改看 MA\n    if ma is None or ma <= prior_high:\n        ratchet, why = prior_high, f"跌破前高 {prior_high:,.0f}（MA{cfg.exit.ma_period} 尚未站上）"\n    else:\n        ratchet, why = ma, f"跌破 MA{cfg.exit.ma_period} {ma:,.0f}"\n\n    if mode == "ma_ratchet":\n        return ratchet, why\n    if mode == "both":\n        return ((trail, f"移動停利：自 {peak_since_breakout:,.0f} 回檔 {cfg.exit.trail_drawdown:.0%}")\n                if trail > ratchet else (ratchet, why))\n    raise ValueError(f"未知的 exit_mode: {mode!r}")\n\n\ndef ladder_weights(cfg: StrategyConfig, target: float) -> list[tuple[float, float]]:\n    """回檔加碼梯的實際權重。\n\n    底倉與加碼梯合計必須剛好等於目標水位：加碼梯的設定值是「彼此之間的比例」，\n    實際可用的額度是 1 − base_weight。少了這一步，base_weight=1.0 會在滿倉之後\n    再加 60%，把部位推到目標的 160%，直接突破風險預算。\n    """\n    remaining = max(0.0, 1.0 - cfg.entry.base_weight)\n    total = sum(w for _, w in cfg.entry.pullback_ladder)\n    if remaining <= 0 or total <= 0:\n        return []\n    return [(thr, w / total * remaining * target) for thr, w in cfg.entry.pullback_ladder]\n\n\ndef _risk_scale(entry_index: float, fill_index: float, levels: Levels,\n                cfg: StrategyConfig) -> float:\n    """往上加碼時的權重縮放。\n\n    部位大小是在訊號日算好的；若之後在更高的價位補倉，同樣的股數要承擔更長的\n    停損距離，實際風險就會超出預算。這裡按停損距離等比縮小，並限制在 1.0 以內\n    ——只會因為買貴而縮手，不會因為買便宜而放大部位。\n    """\n    base = _blended_stop_distance(entry_index, levels, cfg)\n    now = _blended_stop_distance(fill_index, levels, cfg)\n    return min(1.0, base / now) if now > 0 else 1.0\n\n\nclass Engine:\n    """單一部位、單一標的（台灣50正2）的回測 / 實盤訊號引擎。"""\n\n    def __init__(self, cfg: StrategyConfig | None = None):\n        self.cfg = cfg or DEFAULT_CONFIG\n\n    def run(self, bars: list[Bar], etf_prices: list[float]) -> Result:\n        cfg = self.cfg\n        if len(bars) != len(etf_prices):\n            raise ValueError("指數日線與 ETF 價格長度不一致")\n\n        setups = detect_setups(bars, cfg.setup)\n        by_trigger = {s.trigger_index: s for s in setups}\n        mas = (moving_average(bars, cfg.exit.ma_period)\n               if cfg.exit.exit_mode in ("ma_ratchet", "both") else [None] * len(bars))\n\n        cash, units = 1.0, 0.0\n        equity_curve: list[tuple[date, float]] = []\n        trades: list[Trade] = []\n\n        trade: Trade | None = None\n        pending: list[tuple[str, float, str]] = []   # (side, weight, reason)\n        base_equity = 1.0\n        swing_high = 0.0\n        peak_since_breakout = 0.0\n        unfilled: list[tuple[float, float]] = []\n        derisked = reloaded = took_profit = False\n\n        for i, bar in enumerate(bars):\n            px = etf_prices[i]\n\n            # ---- 1. 執行前一交易日收盤掛出的委託 ----\n            for side, weight, reason in pending:\n                if side == "buy":\n                    notional = weight * base_equity\n                    if notional <= 0:\n                        continue\n                    fee = notional * cfg.cost.buy_cost\n                    u = notional / px\n                    cash -= notional + fee\n                    units += u\n                    assert trade is not None\n                    trade.fills.append(\n                        Fill(bar.d, "buy", reason, bar.close, px, weight, u, -(notional + fee))\n                    )\n                    if trade.entry_date is None:\n                        trade.entry_date = bar.d\n                        trade.equity_at_entry = base_equity\n                else:\n                    u = units * weight\n                    if u <= 0:\n                        continue\n                    proceeds = u * px * (1 - cfg.cost.sell_cost)\n                    cash += proceeds\n                    units -= u\n                    assert trade is not None\n                    trade.fills.append(\n                        Fill(bar.d, "sell", reason, bar.close, px, weight, u, proceeds)\n                    )\n            pending = []\n            equity = cash + units * px\n\n            # ---- 2. 收盤後評估，掛出明日委託 ----\n            if trade is None:\n                setup = by_trigger.get(i)\n                if setup is not None:\n                    levels = build_levels(setup.peak, setup.trough, cfg.levels)\n                    target = position_size(bar.close, levels, cfg)\n                    trade = Trade(setup=setup, levels=levels, target_weight=target,\n                                  equity_at_entry=equity)\n                    base_equity = equity\n                    swing_high = bar.close\n                    peak_since_breakout = 0.0\n                    derisked = reloaded = took_profit = False\n                    unfilled = ladder_weights(cfg, target)\n                    pending.append(("buy", cfg.entry.base_weight * target, "底倉：訊號確認，不等回檔"))\n            else:\n                lv, c = trade.levels, bar.close\n                swing_high = max(swing_high, c)\n                bars_since = i - trade.setup.trigger_index\n                trade.swing_high = swing_high\n                trade.bars_held = bars_since\n                if trade.fills:\n                    trade.max_adverse_index = min(\n                        trade.max_adverse_index, c / trade.fills[0].index_price - 1.0\n                    )\n                zone = lv.zone(c)\n\n                if zone == "invalidated" and cfg.exit.hard_stop_at_trough:\n                    unfilled = []\n                    pending.append(("sell", 1.0, f"劇本失效：收盤跌破谷底 {lv.invalidation:,.0f}"))\n                    trade.exit_reason = "stop_trough"\n\n                elif zone == "warning" and not derisked:\n                    unfilled = []\n                    derisked = True\n                    verb = "清倉" if cfg.exit.warn_derisk_fraction >= 1.0 else "減碼"\n                    pending.append(("sell", cfg.exit.warn_derisk_fraction,\n                                    f"警戒{verb}：收盤跌破 {lv.warn_line:,.0f}"\n                                    f"（{cfg.levels.warn_line_ratio:.1%} 回補位）"))\n\n                else:\n                    adds_allowed = zone in ("healthy", "buffer", "breakout")\n\n                    if c >= lv.peak:\n                        trade.reached_prior_high = True\n                        peak_since_breakout = max(peak_since_breakout, c)\n\n                    # (a) 突破前高 → 依設定補齊或取消未成交的分批單\n                    breakout_filled = False\n                    if trade.reached_prior_high and unfilled:\n                        if cfg.entry.breakout_fills_remainder and adds_allowed:\n                            scale = _risk_scale(trade.setup.trigger_close, c, lv, cfg)\n                            for _, w in unfilled:\n                                pending.append(("buy", w * scale,\n                                                f"突破補齊：站上前高 {lv.peak:,.0f}"\n                                                f"（風險縮放 {scale:.0%}）"))\n                            breakout_filled = True\n                        unfilled = []\n\n                    # (b) 回到前高：可選的分批落袋（預設 0，前高不是賣出的理由）\n                    if (trade.reached_prior_high and not took_profit and units > 0\n                            and not breakout_filled and cfg.exit.target_take_fraction > 0):\n                        took_profit = True\n                        pending.append(("sell", cfg.exit.target_take_fraction,\n                                        f"目標達陣：回到前高 {lv.peak:,.0f}"))\n\n                    # (c) 創高之後，出場全部交給停利機制\n                    if trade.reached_prior_high and units > 0:\n                        stop, why = breakout_stop(cfg, lv.peak, peak_since_breakout, mas[i])\n                        if c < stop:\n                            pending.append(("sell", 1.0, why))\n                            trade.exit_reason = "trail"\n\n                    # (d) 減碼後收復主防線 → 補回一次\n                    if derisked and not reloaded and adds_allowed and c >= lv.half_line and units > 0:\n                        reloaded = True\n                        # 解除減碼旗標：回補之後若再度跌破警戒線，還要能再減一次\n                        derisked = False\n                        pending.append(("buy", trade.filled_weight * cfg.exit.warn_derisk_fraction,\n                                        f"回補：收復主防線 {lv.half_line:,.0f}"))\n\n                    # (e) 回檔加碼梯\n                    if unfilled and adds_allowed:\n                        pullback = c / swing_high - 1.0\n                        still: list[tuple[float, float]] = []\n                        for thr, w in unfilled:\n                            if pullback <= -thr:\n                                pending.append(("buy", w, f"回檔加碼：自波段高點 {pullback:.1%}"))\n                            else:\n                                still.append((thr, w))\n                        unfilled = still\n\n                    # (f) 時間補齊：等不到回檔，不參與才是最大的風險\n                    if unfilled and adds_allowed and bars_since >= cfg.entry.fill_timeout_bars:\n                        scale = _risk_scale(trade.setup.trigger_close, c, lv, cfg)\n                        for _, w in unfilled:\n                            pending.append(("buy", w * scale,\n                                            f"時間補齊：{bars_since} 個交易日未見回檔"\n                                            f"（風險縮放 {scale:.0%}）"))\n                        unfilled = []\n\n                    # (g) 劇本過期\n                    if (not trade.reached_prior_high and units > 0\n                            and bars_since >= cfg.setup.setup_expiry_bars):\n                        unfilled = []\n                        pending.append(("sell", 1.0, f"劇本過期：{bars_since} 個交易日未創高"))\n                        trade.exit_reason = "expired"\n\n                trade.pending_ladder = list(unfilled)\n\n                # 部位歸零且沒有待買單 → 結案\n                if units <= 0 and trade.entry_date is not None and not any(\n                        s == "buy" for s, _, _ in pending):\n                    if trade.exit_reason == "open":\n                        trade.exit_reason = "flat"\n                    trade.exit_date = bar.d\n                    trade.equity_at_exit = equity\n                    trades.append(trade)\n                    trade = None\n\n            equity_curve.append((bar.d, cash + units * px))\n\n        if trade is not None:\n            trade.exit_date = bars[-1].d\n            trade.equity_at_exit = cash + units * etf_prices[-1]\n            trades.append(trade)\n\n        return Result(trades=trades, equity_curve=equity_curve, setups=setups)\n'
SOURCES['backtest'] = '"""回測與績效統計。"""\n\nfrom __future__ import annotations\n\nimport statistics\nfrom dataclasses import dataclass\n\nfrom .bars import Bar\nfrom .config import DEFAULT_CONFIG, StrategyConfig\nfrom .engine import Engine, Result\nfrom .leveraged import synth_leveraged_path\n\n\n@dataclass(frozen=True)\nclass Stats:\n    n_setups: int\n    n_trades: int\n    hit_prior_high: int\n    hit_rate: float\n    win_rate: float\n    avg_return: float\n    median_return: float\n    best: float\n    worst: float\n    total_return: float\n    max_drawdown: float\n    median_max_adverse: float\n\n    def render(self) -> str:\n        return "\\n".join([\n            f"訊號次數            {self.n_setups}",\n            f"實際交易            {self.n_trades}",\n            f"回到前高            {self.hit_prior_high} ({self.hit_rate:.0%})",\n            f"獲利比例            {self.win_rate:.0%}",\n            f"單筆平均報酬        {self.avg_return:+.1%}",\n            f"單筆中位數報酬      {self.median_return:+.1%}",\n            f"最佳 / 最差         {self.best:+.1%} / {self.worst:+.1%}",\n            f"權益總報酬          {self.total_return:+.1%}",\n            f"權益最大回檔        {self.max_drawdown:.1%}",\n            f"進場後指數最大逆行  {self.median_max_adverse:.1%}（中位數）",\n        ])\n\n\ndef align_etf(bars: list[Bar], etf_bars: list[Bar]) -> tuple[list[Bar], list[float]]:\n    """把 ETF 日線對齊到指數日線，只保留兩邊都有交易的日子。\n\n    00631L 2014-10-31 才掛牌，比加權指數短很多；用真實 ETF 價格回測時，\n    回測期間會自動縮到重疊區間，而不是拿合成價去補前面那一段。\n    """\n    etf_by_date = {b.d: b.close for b in etf_bars}\n    kept = [(b, etf_by_date[b.d]) for b in bars if b.d in etf_by_date]\n    if not kept:\n        raise ValueError("指數與 ETF 日線沒有重疊的交易日")\n    return [b for b, _ in kept], [p for _, p in kept]\n\n\ndef run_backtest(bars: list[Bar], cfg: StrategyConfig | None = None,\n                 etf_prices: list[float] | None = None) -> tuple[Result, Stats]:\n    cfg = cfg or DEFAULT_CONFIG\n    if etf_prices is None:\n        etf_prices = synth_leveraged_path(bars, cfg.cost, cfg.sizing.leverage)\n    result = Engine(cfg).run(bars, etf_prices)\n    return result, summarize(result)\n\n\ndef summarize(result: Result) -> Stats:\n    rets = [t.ret for t in result.trades]\n    hits = sum(1 for t in result.trades if t.reached_prior_high)\n    adverse = [t.max_adverse_index for t in result.trades] or [0.0]\n\n    peak = -1e18\n    max_dd = 0.0\n    for _, eq in result.equity_curve:\n        peak = max(peak, eq)\n        max_dd = min(max_dd, eq / peak - 1.0)\n\n    n = len(result.trades)\n    return Stats(\n        n_setups=len(result.setups),\n        n_trades=n,\n        hit_prior_high=hits,\n        hit_rate=hits / n if n else 0.0,\n        win_rate=sum(1 for r in rets if r > 0) / n if n else 0.0,\n        avg_return=statistics.fmean(rets) if rets else 0.0,\n        median_return=statistics.median(rets) if rets else 0.0,\n        best=max(rets) if rets else 0.0,\n        worst=min(rets) if rets else 0.0,\n        total_return=result.final_equity - 1.0,\n        max_drawdown=max_dd,\n        median_max_adverse=statistics.median(adverse),\n    )\n'
SOURCES['plan'] = '"""把訊號翻譯成可以直接下單的操作計畫。"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nfrom .config import DEFAULT_CONFIG, StrategyConfig\nfrom .engine import ladder_weights, position_size\nfrom .levels import Levels, build_levels\n\n\n@dataclass(frozen=True)\nclass Order:\n    tag: str\n    weight: float          # 佔總權益比例\n    trigger: str\n    index_level: float | None   # 觸發時的指數概略位置（回檔梯以波段高點推算）\n\n\n@dataclass(frozen=True)\nclass TradePlan:\n    levels: Levels\n    reference_index: float\n    target_weight: float\n    orders: list[Order]\n    cfg: StrategyConfig\n    capital: float | None = None\n\n    def render(self) -> str:\n        lv = self.levels\n        out: list[str] = []\n        out.append("=" * 68)\n        out.append("台灣指數快速修復 → 台灣50正2(00631L) 回檔入場計畫")\n        out.append("=" * 68)\n        out.append("")\n        out.append(f"前波高點 P        {lv.peak:>10,.0f}   ← 主目標（歷史 89% 會回到這裡）")\n        out.append(f"波段谷底 T        {lv.trough:>10,.0f}   ← 失效線：收盤跌破 = 全部出場")\n        out.append(f"跌幅 R            {lv.drop:>10,.0f} 點 ({lv.drop_pct:.1%})")\n        out.append("")\n        ex, lc = self.cfg.exit, self.cfg.levels\n        derisk = "全部出場" if ex.warn_derisk_fraction >= 1.0 else f"減碼 {ex.warn_derisk_fraction:.0%}"\n        out.append("關鍵價位")\n        out.append(f"  61.8% 回補      {lv.trough + 0.618 * lv.drop:>10,.0f}")\n        if abs(lc.warn_line_ratio - lc.half_line_ratio) < 1e-9:\n            # 警戒線與主防線重合：只印一條，避免看起來像兩個不同的動作\n            out.append(f"  主防線 = 警戒線 {lv.half_line:>10,.0f}   "\n                       f"({lc.half_line_ratio:.1%} 回補位) 收盤跌破 → {derisk}")\n            out.append(f"  容忍緩衝 (-{lc.half_line_buffer:.1%}){lv.half_line_with_buffer:>8,.0f}   "\n                       f"假跌破區，不動作")\n        else:\n            out.append(f"  主防線 ({lc.half_line_ratio:.0%})    {lv.half_line:>10,.0f}   歷史回檔多數止步於此")\n            out.append(f"  容忍緩衝 (-{lc.half_line_buffer:.1%}){lv.half_line_with_buffer:>8,.0f}   "\n                       f"假跌破區，不加碼也不減碼")\n            out.append(f"  警戒線 ({lc.warn_line_ratio:.1%}) {lv.warn_line:>10,.0f}   收盤跌破 → {derisk}")\n        out.append(f"  失效線          {lv.invalidation:>10,.0f}   收盤跌破 → 清倉")\n        out.append("")\n        out.append(f"現價（參考）      {self.reference_index:>10,.0f}   "\n                   f"補回 {lv.repair_fraction(self.reference_index):.0%}，"\n                   f"距前高 {self.reference_index / lv.peak - 1:+.1%}，分區：{lv.zone(self.reference_index)}")\n        out.append("")\n        out.append(f"目標總持股水位    {self.target_weight:.0%} 的權益"\n                   + (f"（約 {self.capital * self.target_weight:,.0f} 元）" if self.capital else ""))\n        out.append("")\n        out.append("進場梯（權重為佔總權益比例）"\n                   if len(self.orders) > 1 else "進場（權重為佔總權益比例）")\n        for o in self.orders:\n            if o.weight > 0:\n                size = f"{o.weight:>6.1%}"\n                amount = f"  ≈ {self.capital * o.weight:,.0f} 元" if self.capital else ""\n            else:\n                size, amount = "  剩餘", ""\n            level = f"（指數約 {o.index_level:,.0f}）" if o.index_level else ""\n            out.append(f"  {o.tag:<10} {size}{amount:<18} {o.trigger}{level}")\n        out.append("")\n        out.append("出場")\n        if ex.target_take_fraction > 0:\n            out.append(f"  觸及 {lv.peak:,.0f}    → 賣出 {ex.target_take_fraction:.0%} 落袋，其餘轉停利")\n        else:\n            out.append(f"  觸及 {lv.peak:,.0f}    → 不賣（前高不是賣出的理由），啟動停利")\n        if ex.exit_mode == "trail":\n            out.append(f"  移動停利          自波段最高收盤回檔 {ex.trail_drawdown:.0%}（指數）→ 出清")\n        elif ex.exit_mode == "ma_ratchet":\n            out.append(f"  均線棘輪          出場線 = max(前高, MA{ex.ma_period})，跌破 → 出清")\n        else:\n            out.append(f"  停利（取較緊）    max(前高, MA{ex.ma_period}) 與 "\n                       f"自高點回檔 {ex.trail_drawdown:.0%} 兩者取高 → 跌破出清")\n        out.append(f"  收盤 < {lv.warn_line:,.0f}   → {derisk}")\n        out.append(f"  收盤 < {lv.invalidation:,.0f}   → 全部出場，不留倉")\n        out.append("")\n        out.append("提醒：00631L 為 2 倍槓桿，指數 -1% ≈ ETF -2%（另有波動耗損與內扣）。")\n        to_stop = lv.invalidation / self.reference_index - 1\n        to_warn = lv.warn_line / self.reference_index - 1\n        out.append(f"      現價到警戒線 {to_warn:.1%}（ETF 約 {2 * to_warn:.0%}）、"\n                   f"到失效線 {to_stop:.1%}（ETF 約 {2 * to_stop:.0%}）。")\n        return "\\n".join(out)\n\n\ndef build_plan(peak: float, trough: float, reference_index: float,\n               cfg: StrategyConfig | None = None,\n               capital: float | None = None) -> TradePlan:\n    cfg = cfg or DEFAULT_CONFIG\n    lv = build_levels(peak, trough, cfg.levels)\n    target = position_size(reference_index, lv, cfg)\n\n    orders = [Order(tag="底倉", weight=cfg.entry.base_weight * target,\n                    trigger="訊號確認後次一交易日市價買進，不等回檔", index_level=None)]\n    for thr, w in ladder_weights(cfg, target):\n        orders.append(\n            Order(tag=f"回檔 -{thr:.0%}", weight=w,\n                  trigger=f"自訊號後波段最高收盤回檔 {thr:.0%} 且收盤仍在 {lv.half_line_with_buffer:,.0f} 之上",\n                  index_level=reference_index * (1 - thr))\n        )\n    if len(orders) > 1:      # 只有存在加碼梯時，時間補齊才有意義\n        orders.append(\n            Order(tag="時間補齊", weight=0.0,\n                  trigger=f"訊號後 {cfg.entry.fill_timeout_bars} 個交易日仍未觸發回檔梯 → 剩餘部位市價補齊",\n                  index_level=None)\n        )\n    return TradePlan(levels=lv, reference_index=reference_index, target_weight=target,\n                     orders=orders, cfg=cfg, capital=capital)\n'
SOURCES['status'] = '"""目前這一輪劇本的即時狀態：手上該做什麼、下一個觸發點在哪。"""\n\nfrom __future__ import annotations\n\nfrom .bars import Bar\nfrom .config import StrategyConfig\nfrom .engine import Result, Trade\n\n\ndef render_status(result: Result, bars: list[Bar], cfg: StrategyConfig,\n                  capital: float | None = None) -> str:\n    last = bars[-1]\n    out: list[str] = ["=" * 72,\n                      f"部位現況（資料截至 {last.d}，指數收盤 {last.close:,.0f}）",\n                      "=" * 72, ""]\n\n    open_trades = [t for t in result.trades if t.exit_reason == "open"]\n    if not open_trades:\n        if result.setups:\n            s = result.setups[-1]\n            out.append("目前沒有進行中的部位。最近一次訊號：")\n            out.append(f"  {s.describe()}")\n            if result.trades:\n                t = result.trades[-1]\n                out.append(f"  該筆已於 {t.exit_date} 出場（{t.exit_reason}），報酬 {t.ret:+.1%}")\n        else:\n            out.append("目前沒有進行中的部位，資料期間內也沒有觸發過訊號。")\n        out.append("")\n        out.append("下一次進場條件：自高點回檔 ≥"\n                   f"{cfg.setup.min_drawdown:.0%} 後，谷底起 ≤{cfg.setup.max_repair_bars} 個交易日內"\n                   f"收盤補回 ≥{cfg.setup.repair_fraction:.0%}。")\n        return "\\n".join(out)\n\n    t = open_trades[-1]\n    lv = t.levels\n    c = last.close\n    zone = lv.zone(c)\n\n    out.append(f"訊號        {t.setup.describe()}")\n    out.append(f"已持有      {t.bars_held} 個交易日"\n               f"（劇本有效期 {cfg.setup.setup_expiry_bars} 日）")\n    out.append("")\n    out.append(f"目標水位    {t.target_weight:.1%}"\n               + (f"（約 {capital * t.target_weight:,.0f} 元）" if capital else ""))\n    out.append(f"已建立      {t.filled_weight:.1%}"\n               + (f"（約 {capital * t.filled_weight:,.0f} 元）" if capital else "")\n               + f"\u3000＝ 目標的 {t.filled_weight / t.target_weight:.0%}")\n    out.append("")\n    out.append("已成交")\n    for f in t.fills:\n        out.append("  " + f.describe())\n    out.append("")\n\n    zone_note = {\n        "breakout": "已站上前高 —— 出場交給移動停利",\n        "healthy": "劇本正常 —— 回檔就是加碼機會",\n        "buffer": "主防線假跌破容忍區 —— 不加碼、也不減碼",\n        "caution": "已跌破主防線 —— 停止加碼，只留現有部位",\n        "warning": "已跌破警戒線 —— 減碼一半",\n        "invalidated": "已跌破谷底 —— 劇本失效，清倉",\n    }[zone]\n    out.append(f"目前分區    {zone}\u3000{zone_note}")\n    out.append(f"距前高      {c / lv.peak - 1:+.1%}\u3000"\n               f"距主防線 {c / lv.half_line - 1:+.1%}\u3000"\n               f"距警戒線 {c / lv.warn_line - 1:+.1%}\u3000"\n               f"距失效線 {c / lv.invalidation - 1:+.1%}")\n    out.append("")\n\n    out.append("下一步（收盤價判定，次一交易日執行）")\n    if t.pending_ladder:\n        for thr, w in t.pending_ladder:\n            trigger_px = t.swing_high * (1 - thr)\n            gap = trigger_px / c - 1\n            blocked = trigger_px < lv.half_line_with_buffer\n            note = "\u3000⚠ 觸發價已低於假跌破緩衝線，屆時不執行" if blocked else ""\n            amount = f"（約 {capital * w:,.0f} 元）" if capital else ""\n            out.append(f"  買 {w:>6.1%}{amount}\u3000收盤 ≤ {trigger_px:,.0f}"\n                       f"（自波段高 {t.swing_high:,.0f} 回檔 {thr:.0%}，距現價 {gap:+.1%}）{note}")\n        remaining = cfg.entry.fill_timeout_bars - t.bars_held\n        if remaining > 0:\n            out.append(f"  買 {\'剩餘\':>6}\u3000再過 {remaining} 個交易日仍未觸發回檔梯 → 市價補齊")\n        else:\n            out.append(f"  買 {\'剩餘\':>6}\u3000已過時間門檻 → 次一交易日市價補齊")\n        if cfg.entry.breakout_fills_remainder:\n            out.append(f"  買 {\'剩餘\':>6}\u3000收盤 > {lv.peak:,.0f}（前高）→ 補齊，權重按停損距離縮放")\n    else:\n        out.append("  加碼梯已全部處理，不再加碼")\n\n    if cfg.exit.target_take_fraction > 0:\n        out.append(f"  賣 {cfg.exit.target_take_fraction:>6.0%}\u3000收盤 ≥ {lv.peak:,.0f}（前高）")\n    if t.reached_prior_high:\n        out.append(f"  賣 {\'全部\':>6}\u3000自波段最高收盤回檔 {cfg.exit.trail_drawdown:.0%}（移動停利已啟動）")\n    else:\n        out.append(f"  賣 {\'全部\':>6}\u3000創高後啟動移動停利（自最高收盤回檔 {cfg.exit.trail_drawdown:.0%}）")\n    out.append(f"  賣 {cfg.exit.warn_derisk_fraction:>6.0%}\u3000收盤 < {lv.warn_line:,.0f}（警戒線）")\n    out.append(f"  賣 {\'全部\':>6}\u3000收盤 < {lv.invalidation:,.0f}（失效線）")\n    out.append("")\n\n    # 實際會把部位清光的第一條線：警戒線設定為全數出場時就是它，否則才是失效線\n    full_exit = (lv.warn_line if cfg.exit.warn_derisk_fraction >= 1.0 else lv.invalidation)\n    lev = cfg.sizing.leverage\n    gap = full_exit / c - 1\n    out.append(f"預期最大損失  跌到出清線 {full_exit:,.0f}（{gap:.1%}），"\n               f"00631L 約 {lev * gap:.0%}；依已建立的 {t.filled_weight:.1%} 部位，"\n               f"權益衝擊約 {t.filled_weight * lev * gap:.1%}")\n    if full_exit != lv.invalidation:\n        gap2 = lv.invalidation / c - 1\n        out.append(f"              （跳空直接摜破失效線 {lv.invalidation:,.0f} 的極端情形："\n                   f"{gap2:.1%}，權益衝擊約 {t.filled_weight * lev * gap2:.1%}）")\n    return "\\n".join(out)\n\n\ndef open_trade(result: Result) -> Trade | None:\n    for t in reversed(result.trades):\n        if t.exit_reason == "open":\n            return t\n    return None\n'
SOURCES['__init__'] = '"""台灣指數「快速修復」回檔入場策略（標的：台灣50正2 / 00631L）。"""\n\nfrom .bars import Bar, load_csv\nfrom .config import DEFAULT_CONFIG, StrategyConfig\nfrom .engine import Engine, Result, Trade, position_size\nfrom .levels import Levels, build_levels\nfrom .plan import TradePlan, build_plan\nfrom .setup import FastRepairSetup, detect_setups\n\n__all__ = [\n    "Bar", "load_csv",\n    "StrategyConfig", "DEFAULT_CONFIG",\n    "Levels", "build_levels",\n    "FastRepairSetup", "detect_setups",\n    "Engine", "Result", "Trade", "position_size",\n    "TradePlan", "build_plan",\n]\n'

for _name, _src in SOURCES.items():
    (PKG / f'{_name}.py').write_text(_src, encoding='utf-8')

print(f'已寫入 {len(SOURCES)} 個模組到 tw_backdraw/')


## 4. 取得資料

| 資料 | 來源 | 說明 |
|---|---|---|
| 加權指數 OHLC | `taiex_total_index:*` | 名稱易誤會，實際是**價格指數**，已與證交所核對相符 |
| 00631L | `etl:adj_*` | **還原股價**。00631L 於 2026-03-31 做過約 23:1 分割，用未還原的 `price:收盤價` 會出現 −95.7% 的假單日報酬 |


In [ ]:
import pandas as pd
from finlab import data
from tw_backdraw.bars import Bar
from tw_backdraw.backtest import align_etf

SYMBOL = '00631L'

idx = pd.DataFrame({
    'open':  data.get('taiex_total_index:開盤指數').iloc[:, 0],
    'high':  data.get('taiex_total_index:最高指數').iloc[:, 0],
    'low':   data.get('taiex_total_index:最低指數').iloc[:, 0],
    'close': data.get('taiex_total_index:收盤指數').iloc[:, 0],
}).dropna(subset=['close'])

etf = pd.DataFrame({
    'close': data.get('etl:adj_close')[SYMBOL],
}).dropna(subset=['close'])

def to_bars(df):
    return [Bar(d=d.date(), open=float(r.get('open', r['close'])),
                high=float(r.get('high', r['close'])),
                low=float(r.get('low', r['close'])), close=float(r['close']))
            for d, r in df.iterrows()]

index_bars, etf_prices = align_etf(to_bars(idx), to_bars(etf))
print(f'加權指數 {idx.index.min().date()} ~ {idx.index.max().date()}（{len(idx)} 根）')
print(f'{SYMBOL}   {etf.index.min().date()} ~ {etf.index.max().date()}（{len(etf)} 根）')
print(f'對齊後共 {len(index_bars)} 根，{index_bars[0].d} ~ {index_bars[-1].d}')


## 5. 選擇參數組

| preset | 訊號 | 出場 | 說明 |
|---|---|---|---|
| `tuned` | 30 日 / 補回 60% | 移動停利 8% | **預設**，以「總報酬 ÷ 最大回檔」從 648,000 組中選出 |
| `post` | 15 日 / 補回 75% | 移動停利 8% | 忠於原始貼文：分批加碼、38.2% 減碼一半 |
| `balanced` | 30 日 / 補回 60% | MA40 棘輪 | 出場線 = max(前高, MA40)，勝率較高、回檔較小 |
| `winrate` | 30 日 / 補回 60% | MA40，**無停損** | 全網格勝率最高（靠關掉風控換來的），列出僅供對照 |


In [ ]:
from tw_backdraw.config import PRESETS

PRESET = 'tuned'      # ← 想換參數組改這裡
cfg = PRESETS[PRESET]

print(f'訊號   回檔 ≥{cfg.setup.min_drawdown:.0%}，谷底起 ≤{cfg.setup.max_repair_bars} 日'
      f'內補回 ≥{cfg.setup.repair_fraction:.0%}')
print(f'進場   底倉 {cfg.entry.base_weight:.0%} 的目標水位')
print(f'防守   跌破警戒線（{cfg.levels.warn_line_ratio:.1%} 回補位）→ '
      f'賣出 {cfg.exit.warn_derisk_fraction:.0%}')
print(f'出場   {cfg.exit.exit_mode}，移動停利 {cfg.exit.trail_drawdown:.0%} / '
      f'MA{cfg.exit.ma_period}')


## 6. 跑策略，產生 FinLab 要的 position

**時點對齊是這裡最容易錯的地方**：FinLab 的 `position` 日期是**訊號日**，
實際成交落在**次一交易日**（trades 表裡 `entry_sig_date` 與 `entry_date` 差一天），
而引擎的 `Fill.d` 記的是**成交日** —— 權重必須往前挪一根 K 才對得上，
否則整套策略會慢一天進出場。


In [ ]:
from tw_backdraw.engine import Engine
from tw_backdraw.backtest import summarize

result = Engine(cfg).run(index_bars, etf_prices)
own = summarize(result)
print(own.render())

# 成交紀錄 → 每日目標權重（往前挪一根 K，對齊 FinLab 的訊號日語意）
index_of = {b.d: i for i, b in enumerate(index_bars)}
changes = {}
for t in result.trades:
    for f in t.fills:
        changes.setdefault(max(index_of[f.d] - 1, 0), []).append(f)

weights, w = [], 0.0
for i, bar in enumerate(index_bars):
    for f in changes.get(i, []):
        w = w + f.weight if f.side == 'buy' else w * (1.0 - f.weight)
    weights.append(0.0 if w < 1e-9 else w)

dates = pd.to_datetime([b.d for b in index_bars])
position = pd.DataFrame({SYMBOL: weights}, index=dates)
price = pd.DataFrame({SYMBOL: etf_prices}, index=dates)
print(f'\n在市天數 {(position[SYMBOL] > 0).sum()} / {len(position)}'
      f'（{(position[SYMBOL] > 0).mean():.0%}）')


## 7. FinLab 回測 → `report.display()`

成交價直接餵我們清理過的還原股價，避免 FinLab 內建價格表裡休市日的 NaN 列
讓成交日落空。成本用 ETF 稅率：手續費 0.1425%×0.6 折、證交稅 0.1%（非股票的 0.3%）。


In [ ]:
from finlab.backtest import sim

report = sim(
    position,
    trade_at_price=price,
    position_limit=1,
    fee_ratio=cfg.cost.buy_cost,
    tax_ratio=cfg.cost.tax_rate,
    name=f'加權指數快速修復 {PRESET}（{SYMBOL}）',
    upload=False,
)
report.display()


## 8. 驗證：FinLab 與內建引擎逐筆對齊

兩邊的進出場日期應該完全一致。
報酬欄位定義不同：FinLab 的 `return` 是**個股報酬**，引擎的 `Trade.ret` 是
**權益報酬**，單一標的下 `權益 ≈ 個股 × 目標水位`。


In [ ]:
trades = report.trades.reset_index()
print(f"{'進場':<12}{'出場':<12}{'引擎(權益)':>12}{'FinLab(個股)':>14}   對齊")
mismatch = 0
for i, t in enumerate(result.trades):
    if i >= len(trades):
        break
    f = trades.iloc[i]
    f_in = f['entry_date'].date()
    f_out = f['exit_date'].date() if f['exit_date'] == f['exit_date'] else None
    own_in = t.fills[0].d if t.fills else None
    own_out = t.fills[-1].d if len(t.fills) > 1 else None
    ok = (f_in == own_in) and (f_out == own_out)
    mismatch += not ok
    print(f"{own_in!s:<12}{str(own_out or '持有中'):<12}{t.ret:>12.1%}"
          f"{f['return']:>14.1%}   {'✓' if ok else '✗'}")
print(f"\n{'✓ 全部一致' if not mismatch else f'⚠ 有 {mismatch} 筆不一致'}")


## 9. 與買進持有對照

**這張表是判斷策略有沒有價值的關鍵。** 策略與對照組使用同一套指標公式
（FinLab 的 `daily_sharpe` 定義與此不同，混用會變成蘋果比橘子）。


In [ ]:
import numpy as np

def series_metrics(s):
    ret = s.pct_change().dropna()
    years = (s.index[-1] - s.index[0]).days / 365.25
    total = float(s.iloc[-1] / s.iloc[0] - 1)
    mdd = float((s / s.cummax() - 1).min())
    cagr = (1 + total) ** (1 / years) - 1
    down = ret[ret < 0].std()
    return dict(CAGR=cagr, 總報酬=total, 最大回檔=mdd,
                Sharpe=float(ret.mean() / ret.std() * np.sqrt(252)),
                Sortino=float(ret.mean() / down * np.sqrt(252)) if down else np.nan,
                Calmar=cagr / abs(mdd) if mdd else np.nan)

lo, hi = position.index[0], position.index[-1]
rows = {f'策略 {PRESET}': series_metrics(report.creturn)}
for b, src in ((SYMBOL, 'etl:adj_close'), ('0050', 'etl:adj_close')):
    s = data.get(src)[b].dropna()
    s = s[(s.index >= lo) & (s.index <= hi)]
    rows[f'買進持有 {b}'] = series_metrics(s)

table = pd.DataFrame(rows).T
for c in ('CAGR', '總報酬', '最大回檔'):
    table[c] = table[c].map('{:.1%}'.format)
for c in ('Sharpe', 'Sortino', 'Calmar'):
    table[c] = table[c].map('{:.2f}'.format)
table


## 10. 目前部位該做什麼

印出進行中部位的已建立水位、目前分區，以及每一個尚未觸發的加碼／減碼價位。
`--capital` 換算金額，改下面的 `CAPITAL` 即可。


In [ ]:
from tw_backdraw.status import render_status

CAPITAL = 1_000_000
print(render_status(result, index_bars, cfg, capital=CAPITAL))


## 11. 換參數做敏感度測試

所有參數都在 `tw_backdraw/config.py`，可以用 `dataclasses.replace` 局部覆寫。


In [ ]:
from dataclasses import replace
from tw_backdraw.backtest import run_backtest

print(f"{'移動停利':>8}{'筆數':>6}{'勝率':>7}{'總報酬':>11}{'最大回檔':>10}{'報酬/回檔':>11}")
for trail in (0.06, 0.08, 0.10, 0.12, 0.15):
    c = replace(cfg, exit=replace(cfg.exit, trail_drawdown=trail))
    _, st = run_backtest(index_bars, c, etf_prices)
    ratio = st.total_return / abs(st.max_drawdown) if st.max_drawdown else 0
    print(f'{trail:>8.0%}{st.n_trades:>6}{st.win_rate:>7.0%}'
          f'{st.total_return:>11.1%}{st.max_drawdown:>10.1%}{ratio:>11.1f}')


---
## 已知限制

1. **樣本數極少。** 00631L 2014-10 才掛牌，回測期間僅約 7 次訊號。
   預設參數是從這批資料選出來的，屬樣本內結果。
2. **獲利集中。** 以加權指數全期（1999 起）拆解，2010–2019 十年累積僅 0.84x，
   獲利幾乎全部集中在 2020 之後。這組參數本質上是在押注暴力單邊行情。
3. **CAGR 輸給買進持有。** 見第 9 節 —— 策略買的是**風險調整後報酬**
   （回檔砍半、Sharpe 與 Calmar 較佳），不是絕對報酬。
4. **跳空風險。** 「跌破警戒線全部出場」在跳空時無法保證執行。

完整討論見 repo 的 `docs/strategy.md`。
